In [ ]:
%%writefile vector_add.cu 

#include <iostream>          // Input Output
#include <cuda_runtime.h>   // CUDA functions
#include <limits>           // numeric_limits

using namespace std;


// ================= CUDA KERNEL =================

// GPU function
__global__ void add(int* A, int* B, int* C, int size) {

    // Unique thread ID
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    // Check valid index
    if (tid < size) {

        // Vector Addition
        C[tid] = A[tid] + B[tid];
    }
}


// ================= PRINT FUNCTION =================

// Display vector
void print(int* vec, int size) {

    for (int i = 0; i < size; i++) {
        cout << vec[i] << " ";
    }

    cout << endl;
}


// ================= SAFE INTEGER INPUT =================

// Function for valid integer input
int getInt(string message) {

    int value;

    while (true) {

        cout << message;

        cin >> value;

        // Check invalid input
        if (cin.fail()) {

            cout << "Invalid input! Enter a number.\n";

            cin.clear();

            cin.ignore(numeric_limits<streamsize>::max(), '\n');
        }

        // Check positive number
        else if (value <= 0) {

            cout << "Value must be > 0.\n";
        }

        else {

            return value;
        }
    }
}


// ================= VECTOR INPUT =================

// Function to input vector elements
void inputVector(int* vec, int size, string name) {

    cout << "Enter elements of " << name << ":\n";

    for (int i = 0; i < size; i++) {

        while (true) {

            cout << name << "[" << i << "] = ";

            cin >> vec[i];

            // Invalid input check
            if (cin.fail()) {

                cout << "Invalid input! Enter a number.\n";

                cin.clear();

                cin.ignore(numeric_limits<streamsize>::max(), '\n');
            }

            else {

                break;
            }
        }
    }
}


// ================= MAIN FUNCTION =================

int main() {

    int N;

    // Input vector size
    N = getInt("Enter size of vectors: ");

    // Total bytes required
    size_t bytes = N * sizeof(int);

    // Host memory allocation (CPU)
    int *A = new int[N];
    int *B = new int[N];
    int *C = new int[N];

    // Input vectors
    inputVector(A, N, "A");

    inputVector(B, N, "B");

    // Print vectors
    cout << "\nVector A: ";
    print(A, N);

    cout << "Vector B: ";
    print(B, N);


    // ================= DEVICE MEMORY =================

    // GPU memory pointers
    int *d_A, *d_B, *d_C;

    // Allocate memory on GPU
    cudaMalloc(&d_A, bytes);

    cudaMalloc(&d_B, bytes);

    cudaMalloc(&d_C, bytes);


    // Copy data CPU -> GPU
    cudaMemcpy(d_A, A, bytes, cudaMemcpyHostToDevice);

    cudaMemcpy(d_B, B, bytes, cudaMemcpyHostToDevice);


    // ================= THREAD CONFIGURATION =================

    // Threads in one block
    int threadsPerBlock = 256;

    // Number of blocks
    int blocksPerGrid =
    (N + threadsPerBlock - 1) / threadsPerBlock;


    // ================= KERNEL LAUNCH =================

    add<<<blocksPerGrid, threadsPerBlock>>>
    (d_A, d_B, d_C, N);


    // Wait until GPU finishes
    cudaDeviceSynchronize();


    // Copy result GPU -> CPU
    cudaMemcpy(C, d_C, bytes,
               cudaMemcpyDeviceToHost);


    // Print result
    cout << "Addition Result: ";

    print(C, N);


    // ================= MEMORY CLEANUP =================

    // Free CPU memory
    delete[] A;
    delete[] B;
    delete[] C;

    // Free GPU memory
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Writing vector_add.cu


In [ ]:
!nvcc vector_add.cu -o vector_add
!./vector_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Enter size of vectors: 5
Enter elements of A:
A[0] = 11
A[1] = 33
A[2] = 13
A[3] = 65
A[4] = 23
Enter elements of B:
B[0] = 43
B[1] = 23
B[2] = 45
B[3] = 83
B[4] = 21

Vector A: 11 33 13 65 23 
Vector B: 43 23 45 83 21 
Addition Result: 54 56 58 148 44 
